# ramai — play Rami against an AI that sees the cards

**Author:** Amine Harch El Korane  
**License:** MIT  
**Repo:** https://github.com/VitalCheffe/ramai  
**Tests:** 131 passing

---

## What this is

An AI that plays Moroccan Rami against you by **looking at your real cards on your real table** with a camera. Point your iPad at the table, the AI sees the discard pile, the melds, and the cards it has shown you. Your hand stays in your hand — you enter it once at the start, then once per draw from the stock.

## Target hardware

iPad on Safari, Colab notebook open in the Colab app. The camera popup is requested at startup. If it doesn't appear (Safari blocks getUserMedia in iframes), the notebook automatically falls back to the native camera app via `<input type=file capture=environment>` — you press a button, the iPad camera app opens, you take a photo, the photo comes back to the notebook.

## Game flow

1. **Calibration** — you photograph the empty table, then 3 test cards. The notebook confirms it sees them. No game starts until calibration passes.
2. **Initial hand entry** — you click your 14 cards in a grid. This happens ONCE.
3. **Each turn** — you draw (stock or discard). If you draw from stock, you click the 1 new card in the grid. If you draw from discard, no entry needed (the camera saw it). The AI plays its turn, you photograph the table state, the AI explains its move in French (its personality), you execute physically.
4. **Detection uncertainty** — if a detection confidence is below threshold, the photo is shown with bounding boxes and you confirm card-by-card. Never silent, never auto-guessed.

---

Run the cells in order.

## Cell 1 — Install, imports, camera pre-warm

This cell installs dependencies and triggers the camera permission prompt immediately. On iPad Safari, if the prompt doesn't appear, the notebook will switch to native camera app fallback in Cell 3.

In [ ]:
!pip install -q ultralytics ipywidgets 2>&1 | tail -3

import os, sys, json, time, random
from pathlib import Path
from IPython.display import display, HTML, Image as IPImage, clear_output
import ipywidgets as widgets
import numpy as np
import cv2
import matplotlib.pyplot as plt

if not Path('/content/ramai').exists():
    !git clone -q https://github.com/VitalCheffe/ramai.git /content/ramai
    
REPO = Path('/content/ramai')
sys.path.insert(0, str(REPO))

from rami.config import RamiConfig
from rami.cards import Card, build_deck, Hand, SUIT_SYMBOLS, RANK_NAMES
from rami.engine import (is_valid_meld, valid_melds, deadwood_score,
                          best_meld_partition, meld_points)
from rami.game import new_game, legal_moves, apply_move, Move, GameState
from rami.extensions import (designate_jokers, find_meld_extensions,
                              all_laid_melds, JokerDesignation)
from rami.counting import CardCountingState
from rami.protocol import (
    ProtocolStep, TurnContext, next_step,
    should_show_ai_hand, show_ai_hand_warning,
)
from rami.ai.discovery import DiscoveryAI
from rami.ai.strategy import StrategyAI
from rami.ai.champion import ChampionAI
from rami.vision import (
    CardDetector, MockDetector, calibrate_camera,
    detect_discard_pile, detect_meld_clusters, find_extendable_melds,
    prewarm_camera, capture_photo, is_camera_ready, is_file_input_mode,
    get_camera_status, init_file_input_mode, capture_photo_file_input,
    try_download_pretrained, get_model_info, get_download_url,
)

print('✓ ramai loaded')
print(f'  Tests: 131 (engine 79 + protocol 40 + camera/manual 12)')
print()
print('--- Camera pre-warm ---')
print('A permission popup should appear in the browser.')
print('If it does NOT appear (iPad Safari), the notebook will switch')
print('to native camera app fallback automatically.')
print()
cam_status = prewarm_camera()
print(f'Camera status: {cam_status}')
if cam_status == 'granted':
    print('  ✓ Camera ready (getUserMedia stream).')
elif cam_status == 'file_input':
    print('  ⚠ getUserMedia blocked (likely iPad Safari in iframe).')
    print('  → Switching to native camera app fallback.')
    print('  → Each capture will open the iPad Camera app.')
    init_file_input_mode()
else:
    print(f'  ✗ Camera unavailable ({cam_status}).')
    print('  → Mode MANUEL will be used (manual card entry, no photos).')
globals()['CAMERA_STATUS'] = cam_status

## Cell 2 — Download YOLO weights + load detector

Downloads the YOLOv8 weights from the project's GitHub release. The release URL is permanent.

The current release (`v0.1.0-vision-bootstrap`) contains the YOLOv8n COCO-pretrained backbone. To get a CARDS-specific model, run `notebooks/train_yolo.ipynb` in Colab (30 min on free GPU), then upload the resulting `best.pt` to a new release `v0.2.0-cards`.

In [ ]:
WEIGHTS = REPO / 'models' / 'yolov8n.pt'
DOWNLOAD_URL = get_download_url()
print(f'Download URL: {DOWNLOAD_URL}')
print()
if not WEIGHTS.exists():
    print('Downloading YOLO weights from GitHub release...')
    result = try_download_pretrained(out_path=str(WEIGHTS), timeout=60)
    if result is None:
        print('✗ Download failed. Check internet connection.')
    else:
        info = get_model_info(str(WEIGHTS))
        print(f'✓ Downloaded: {info["size_mb"]} MB')
else:
    info = get_model_info(str(WEIGHTS))
    print(f'✓ Weights already present: {info["size_mb"]} MB')

# Load detector
detector = MockDetector()
if WEIGHTS.exists():
    try:
        detector = CardDetector(weights_path=str(WEIGHTS))
        print(f'✓ Detector loaded: {type(detector).__name__}')
        print(f'  Classes: {len(detector.names)} classes')
    except Exception as e:
        print(f'⚠ Could not load detector: {e}')
        print('  → Using MockDetector (no detection).')
else:
    print('⚠ No weights — using MockDetector (no detection).')

globals()['DETECTOR'] = detector

## Cell 3 — Game configuration

Choose AI level and Rami variant. Camera mode is always AUTO if the camera is available; the notebook falls back to MANUAL automatically if not.

In [ ]:
ai_level = widgets.Dropdown(
    options=[('Discovery (rules only, AI hand visible)', 'discovery'),
             ('Strategy (perfect card counting, AI hand hidden)', 'strategy'),
             ('Champion (RL self-play, AI hand hidden)', 'champion')],
    value='strategy',
    description='AI level:',
    style={'description_width': 'initial'}
)
variant = widgets.Dropdown(
    options=[('Classic Moroccan (threshold 30)', 'classic'),
             ('Rami 51 (threshold 51, no discard before threshold)', '51'),
             ('No threshold', 'none'),
             ('No jokers', 'nojokers')],
    value='classic',
    description='Variant:',
    style={'description_width': 'initial'}
)
display(ai_level, variant)

confirm = widgets.Button(description='Confirm configuration', button_style='primary')
output_area = widgets.Output()
display(confirm, output_area)

def on_confirm(b):
    output_area.clear_output()
    if variant.value == 'classic':
        cfg = RamiConfig.classic_moroccan()
    elif variant.value == '51':
        cfg = RamiConfig.threshold_51()
    elif variant.value == 'none':
        cfg = RamiConfig.no_threshold()
    else:
        cfg = RamiConfig.no_jokers()
    
    if ai_level.value == 'discovery':
        ai = DiscoveryAI(seed=0)
    elif ai_level.value == 'strategy':
        ai = StrategyAI(seed=0)
    else:
        weights_path = str(REPO / 'models' / 'champion_weights.json')
        if not os.path.exists(weights_path):
            with output_area:
                print('⚠ Champion not trained yet. Training 500 games...')
                !cd {REPO} && python scripts/train_champion.py --games 500 --candidates 6
        ai = ChampionAI(weights_path=weights_path, seed=0)
    
    cam_status = globals().get('CAMERA_STATUS', 'unavailable')
    use_camera = cam_status in ('granted', 'file_input')
    
    with output_area:
        print(f'✓ Configuration confirmed')
        print(f'  AI: {ai.name}')
        print(f'  Variant: {variant.label}')
        print(f'  First meld threshold: {cfg.first_meld_threshold} pts')
        if cfg.block_discard_before_threshold:
            print(f'  Rami 51: NO discard draw before threshold met')
        print(f'  Camera mode: {"AUTO" if use_camera else "MANUAL (debug fallback)"}')
        if cam_status == 'file_input':
            print(f'    (using native camera app — iPad Safari fallback)')
        print(f'  AI hand visible: {should_show_ai_hand(ai_level.value)}')
        globals()['CFG'] = cfg
        globals()['AI'] = ai
        globals()['AI_LEVEL'] = ai_level.value
        globals()['USE_CAMERA'] = use_camera

confirm.on_click(on_confirm)

## Cell 4 — Calibration (REQUIRED before playing)

Photograph the empty table first, then 3 test cards laid out. The notebook must confirm it can see them before any game starts.

If you're in MANUAL mode (no camera), this cell is skipped.

In [ ]:
USE_CAMERA = globals().get('USE_CAMERA', False)
DETECTOR = globals().get('DETECTOR', MockDetector())

calib_out = widgets.Output()
display(calib_out)

if not USE_CAMERA:
    with calib_out:
        print('⚠ MANUAL MODE (debug) — calibration skipped.')
        print('  Camera is not available. You will enter ALL cards manually.')
        print('  This is the debug fallback. The real project uses the camera.')
        globals()['CALIBRATION_OK'] = True
else:
    calib_btn1 = widgets.Button(description='1️⃣ Photo of empty table', button_style='info')
    calib_btn2 = widgets.Button(description='2️⃣ Photo of 3 test cards', button_style='info')
    calib_btn3 = widgets.Button(description='3️⃣ Validate calibration', button_style='success')
    display(widgets.HBox([calib_btn1, calib_btn2, calib_btn3]))
    
    calib_state = {'empty_ok': False, 'cards_ok': False, 'cards_detected': 0}
    globals()['CALIBRATION_OK'] = False
    
    def on_empty(b):
        calib_out.clear_output()
        with calib_out:
            print('📸 Taking photo of empty table...')
            img = capture_photo()
            if img is None:
                print('✗ Capture failed.')
                return
            result = calibrate_camera(img)
            print(f'Angle: {result.tilt_degrees:.1f}°')
            print(result.message)
            if result.is_good:
                calib_state['empty_ok'] = True
                print('✓ Empty table photo OK. Now place 3 cards and click button 2.')
            from google.colab.patches import cv2_imshow
            cv2_imshow(img)
    
    def on_cards(b):
        calib_out.clear_output()
        with calib_out:
            print('📸 Taking photo of 3 test cards...')
            print('Place 3 cards face up in the camera view, then click this button.')
            img = capture_photo()
            if img is None:
                print('✗ Capture failed.')
                return
            detections = DETECTOR.predict(img)
            calib_state['cards_detected'] = len(detections)
            print(f'Detected {len(detections)} cards:')
            for d in detections:
                color = (0, 255, 0) if d.confidence > 0.7 else (0, 165, 255)
                x1, y1, x2, y2 = [int(v) for v in d.bbox]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                cv2.putText(img, f'{d.rank}{d.suit} {d.confidence:.2f}',
                            (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            if len(detections) >= 3:
                calib_state['cards_ok'] = True
                print('✓ Calibration passed — at least 3 cards detected.')
            else:
                print(f'⚠ Only {len(detections)} cards detected. Try better lighting / angle.')
            from google.colab.patches import cv2_imshow
            cv2_imshow(img)
    
    def on_validate(b):
        calib_out.clear_output()
        with calib_out:
            if calib_state['empty_ok'] and calib_state['cards_ok']:
                print('✓ Calibration complete. You can start the game (Cell 5).')
                globals()['CALIBRATION_OK'] = True
            else:
                print('✗ Calibration not complete. Steps:')
                print(f'  Empty table photo: {"OK" if calib_state["empty_ok"] else "MISSING"}')
                print(f'  3 cards detected:   {"OK" if calib_state["cards_ok"] else f"MISSING ({calib_state[\"cards_detected\"]} cards)"}')
    
    calib_btn1.on_click(on_empty)
    calib_btn2.on_click(on_cards)
    calib_btn3.on_click(on_validate)

## Cell 5 — Enter your initial hand (14 cards)

Click the 14 cards you have in your hand. This is the ONLY time you enter all 14 — after this, you only enter 1 card when you draw from the stock (the camera can't see your hand, only you can).

In [ ]:
def make_card_button(rank, suit):
    if rank == 0:
        label = '★'
        color = 'purple'
    else:
        label = f'{RANK_NAMES[rank]}{SUIT_SYMBOLS[suit]}'
        color = 'red' if suit in (1, 2) else 'black'
    btn = widgets.ToggleButton(
        description=label,
        button_style='',
        layout=widgets.Layout(width='50px', height='40px'),
        style={'button_color': 'white', 'font_weight': 'bold'},
    )
    btn.rank = rank
    btn.suit = suit
    btn.color = color
    return btn

def make_hand_selector():
    grid = widgets.GridBox(
        layout=widgets.Layout(
            grid_template_columns='repeat(13, 50px)',
            grid_gap='4px'
        )
    )
    buttons = []
    for suit in range(4):
        for rank in range(1, 14):
            btn = make_card_button(rank, suit)
            buttons.append(btn)
    for j in range(2):
        btn = make_card_button(0, -1)
        buttons.append(btn)
    grid.children = buttons
    return grid, buttons

selected_count = widgets.Label(value='Selected: 0 / 14')
grid, all_buttons = make_hand_selector()
display(widgets.HTML('<b>Your hand (click your 14 cards):</b>'))
display(selected_count, grid)

def update_count():
    n = sum(1 for b in all_buttons if b.value)
    selected_count.value = f'Selected: {n} / 14'

for b in all_buttons:
    b.observe(lambda c: update_count(), 'value')

validate_hand_btn = widgets.Button(description='Confirm hand', button_style='success')
hand_out = widgets.Output()
display(validate_hand_btn, hand_out)

def on_validate_hand(b):
    hand_out.clear_output()
    selected = [Card(suit=b.suit, rank=b.rank, copy_id=0) for b in all_buttons if b.value]
    if len(selected) != 14:
        with hand_out:
            print(f'⚠ You selected {len(selected)} cards. Need exactly 14.')
        return
    globals()['HUMAN_HAND'] = selected
    globals()['HAND_BUTTONS'] = all_buttons
    with hand_out:
        print(f'✓ Hand confirmed: {" ".join(c.name for c in selected)}')

validate_hand_btn.on_click(on_validate_hand)

## Cell 6 — Initialize the game

In [ ]:
CFG = globals().get('CFG', RamiConfig())
AI = globals().get('AI', StrategyAI(seed=0))
AI_LEVEL = globals().get('AI_LEVEL', 'strategy')
USE_CAMERA = globals().get('USE_CAMERA', False)
DETECTOR = globals().get('DETECTOR', MockDetector())

if not globals().get('CALIBRATION_OK', False) and USE_CAMERA:
    print('⚠ Calibration not done! Go back to Cell 4.')
else:
    state = new_game(CFG, seed=int(time.time()) % 1000)
    if 'HUMAN_HAND' in globals():
        state.players[0].hand.cards = list(globals()['HUMAN_HAND'])
    counting = CardCountingState.fresh(
        CFG, ai_player_idx=1,
        ai_hand=state.players[1].hand.cards,
        initial_discard=state.discard,
    )
    print(f'Game started.')
    print(f'  You (P0): {len(state.players[0].hand)} cards in hand (hidden from camera)')
    print(f'  RAMAI (P1, {AI.name}): {len(state.players[1].hand)} cards in hand')
    print(f'  Camera mode: {"AUTO" if USE_CAMERA else "MANUAL (debug)"}')
    print(f'  Top of discard: {state.top_discard.name if state.top_discard else "—"}')
    print(f'  Stock: {len(state.stock)} cards')
    if should_show_ai_hand(AI_LEVEL):
        print(f'  RAMAI hand: {" ".join(c.name for c in state.players[1].hand.cards)}')
        print(f'    (visible — Discovery mode, pedagogical)')

## Cell 7 — Your turn

1. Click "Draw stock" or "Draw discard"
2. If you drew from stock, click the 1 new card in the grid (your hand can't be seen by the camera)
3. (Optional) Lay melds — not yet implemented in this cell, will be added
4. Click the card you want to discard
5. Click "End my turn" — if camera mode, you'll be prompted to photograph the discard pile

In [ ]:
draw_stock_btn = widgets.Button(description='Draw from stock', button_style='info')
draw_discard_btn = widgets.Button(description='Take discard', button_style='warning')
if CFG.block_discard_before_threshold and not state.players[0].has_laid_first:
    draw_discard_btn.disabled = True
    draw_discard_btn.tooltip = 'Rami 51: forbidden before threshold'

end_turn_btn = widgets.Button(description='End my turn', button_style='success')

human_out = widgets.Output()
display(widgets.HBox([draw_stock_btn, draw_discard_btn]), end_turn_btn, human_out)

human_action = {'draw_source': None, 'drawn_card': None,
                'discard_card': None, 'photo_taken': False}

def on_draw_stock(b):
    human_out.clear_output()
    if not state.stock:
        with human_out: print('Stock is empty.')
        return
    drawn = state.stock.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'stock', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'stock'
    human_action['drawn_card'] = drawn
    with human_out:
        print(f'You drew from stock. Click the new card in the grid above.')
        print(f'(Camera cannot see your hand. You must enter the drawn card manually.)')
        print(f'Then click the card you want to discard.')

def on_draw_discard(b):
    human_out.clear_output()
    if not state.discard:
        with human_out: print('No discard pile.')
        return
    drawn = state.discard.pop()
    state.players[0].hand.add(drawn)
    counting.record_draw(0, 'discard', drawn, ai_player_idx=1)
    human_action['draw_source'] = 'discard'
    human_action['drawn_card'] = drawn
    with human_out:
        print(f'You took the discard: {drawn.name}')
        print(f'(The camera saw this card. No manual entry needed.)')
        print(f'Now click the card you want to discard.')

def on_end_turn(b):
    human_out.clear_output()
    if not human_action['draw_source']:
        with human_out: print('You must draw first.')
        return
    # Get the latest clicked card as the discard
    selected = [b for b in all_buttons if b.value]
    if not selected:
        with human_out: print('Click the card you want to discard in the grid above.')
        return
    latest = selected[-1]  # last clicked
    discard = Card(suit=latest.suit, rank=latest.rank, copy_id=0)
    if discard not in state.players[0].hand.cards:
        with human_out: print(f'{discard.name} is not in your hand. Pick another.')
        return
    state.players[0].hand.remove(discard)
    state.discard.append(discard)
    counting.record_discard(0, discard)
    human_action['discard_card'] = discard
    
    with human_out:
        print(f'You discarded: {discard.name}')
        # Photo of discard (mandatory in AUTO mode)
        if USE_CAMERA:
            print()
            print('📸 MANDATORY photo of the discard pile:')
            img = capture_photo()
            if img is not None:
                result = detect_discard_pile(DETECTOR, img)
                print(result.message)
                if not result.is_reliable:
                    print('⚠ Detection uncertain. Showing photo with boxes for confirmation.')
                    # Show photo with boxes for manual confirmation
                    detections = DETECTOR.predict(img)
                    for d in detections:
                        x1, y1, x2, y2 = [int(v) for v in d.bbox]
                        color = (0, 255, 0) if d.confidence > 0.7 else (0, 0, 255)
                        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(img, f'{d.rank}{d.suit} {d.confidence:.2f}',
                                    (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(img)
                    print('Confirm: is the top card really ' + result.card_rank + result.card_suit + '?')
                human_action['photo_taken'] = True
            else:
                print('✗ Capture failed.')
                return
        else:
            human_action['photo_taken'] = True  # manual mode
        
        # Card counting check
        est = counting.opponent_hand_estimate(CFG, opponent_idx=0,
                                                stock_size=len(state.stock))
        print(f'Your hand (inferred by RAMAI): {est["hand_count"]} cards')
        print(f'  Arithmetic: {"OK" if est["arithmetic_consistent"] else "INCONSISTENT"}')
        if counting.is_opponent_empty(0):
            print('🏁 You won!')
            state.winner = 0
            state.terminal = True
            return
        state.current = 1
        state.turn += 1
        print(f'→ RAMAI turn (Cell 8)')

draw_stock_btn.on_click(on_draw_stock)
draw_discard_btn.on_click(on_draw_discard)
end_turn_btn.on_click(on_end_turn)

## Cell 8 — RAMAI's turn

RAMAI announces its decision and explains its reasoning in French (its personality). You execute the move physically on the table, then photograph the resulting state.

In [ ]:
ai_play_btn = widgets.Button(description='🎯 RAMAI plays', button_style='primary')
ai_out = widgets.Output()
display(ai_play_btn, ai_out)

def explain_move(state, move, ai):
    """AI explains its move in French (its personality)."""
    lines = []
    if move.draw_source == 'discard':
        top = state.top_discard
        lines.append(f"Je prends la défausse ({top.name}).")
    else:
        lines.append(f"Je pioche dans le talon (carte inconnue).")
    if move.laydowns:
        lines.append(f"Je pose {len(move.laydowns)} meld(s) :")
        for meld in move.laydowns:
            cards_str = ' '.join(c.name for c in meld)
            lines.append(f"  → {cards_str}")
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                lines.append(f"      {d.name}")
    else:
        lines.append(f"Je ne pose rien ce tour.")
    lines.append(f"Je jette : {move.discard.name}")
    return '\n'.join(lines)

def on_ai_play(b):
    ai_out.clear_output()
    with ai_out:
        if state.terminal:
            print('Game over.')
            return
        if state.current != 1:
            print("It's not RAMAI's turn. Play your turn first (Cell 7).")
            return
        print(f'--- Turn {state.turn + 1} | RAMAI ({AI.name}) ---')
        if should_show_ai_hand(AI_LEVEL):
            print(f'RAMAI hand: {" ".join(c.name for c in state.players[1].hand.cards)}')
        print()
        
        m = AI.decide(state)
        print(explain_move(state, m, AI))
        
        # Apply the move
        if m.draw_source == 'stock':
            drawn = state.stock.pop()
            counting.record_draw(1, 'stock', drawn, ai_player_idx=1)
        else:
            drawn = state.discard.pop()
            counting.record_draw(1, 'discard', drawn, ai_player_idx=1)
        state.players[1].hand.add(drawn)
        
        for meld in m.laydowns:
            for card in meld:
                state.players[1].hand.remove(card)
            state.players[1].laid_melds.append(meld)
            counting.record_meld(1, meld)
            if not state.players[1].has_laid_first:
                state.players[1].has_laid_first = True
        
        state.players[1].hand.remove(m.discard)
        state.discard.append(m.discard)
        counting.record_discard(1, m.discard)
        
        print()
        print(f'Top of discard: {state.top_discard.name}')
        
        # MANDATORY photo of discard if camera mode
        if USE_CAMERA:
            print()
            print('📸 MANDATORY photo of the new discard pile:')
            img = capture_photo()
            if img is not None:
                result = detect_discard_pile(DETECTOR, img)
                print(result.message)
                if not result.is_reliable:
                    print('⚠ Detection uncertain. Showing photo for confirmation.')
                    from google.colab.patches import cv2_imshow
                    cv2_imshow(img)
        
        if counting.is_opponent_empty(1):
            print('🏁 RAMAI won!')
            state.winner = 1
            state.terminal = True
        else:
            state.current = 0
            state.turn += 1
            print()
            print(f'→ Your turn. Top discard: {state.top_discard.name}')

ai_play_btn.on_click(on_ai_play)

## Cell 9 — Cheat button (reveal RAMAI's hand)

Hidden by default. Click twice to confirm.

In [ ]:
triche_btn = widgets.Button(description='👁 Cheat: reveal RAMAI hand', button_style='danger')
triche_out = widgets.Output()
display(triche_btn, triche_out)

triche_confirmed = [False]

def on_triche(b):
    triche_out.clear_output()
    if not triche_confirmed[0]:
        triche_confirmed[0] = True
        with triche_out:
            print(show_ai_hand_warning())
            print('Click again to confirm.')
        return
    triche_confirmed[0] = False
    with triche_out:
        if state.terminal:
            print('Game over.')
            return
        print('⚠ CHEAT — RAMAI hand:')
        print('  ', ' '.join(c.name for c in state.players[1].hand.cards))
        est = counting.opponent_hand_estimate(CFG, opponent_idx=1,
                                                stock_size=len(state.stock))
        print(f'Card counting: {est["hand_count"]} cards inferred, {len(state.players[1].hand)} real')

triche_btn.on_click(on_triche)

## Cell 10 — Meld extensions + joker designations on the table

In [ ]:
ext_out = widgets.Output()
display(ext_out)

with ext_out:
    print('=== Melds on the table ===')
    print()
    all_melds = all_laid_melds(state)
    if not all_melds:
        print('No melds laid yet.')
    else:
        for p_idx, m_idx, meld in all_melds:
            player_name = 'You' if p_idx == 0 else 'RAMAI'
            print(f'{player_name} meld: {" ".join(c.name for c in meld)}')
            desigs = designate_jokers(meld, CFG)
            for d in desigs:
                print(f'  {d.name}')
            print()
    
    ai_hand = state.players[1].hand.cards
    extensions_found = []
    for card in ai_hand:
        if card.is_joker:
            continue
        exts = find_meld_extensions(card, all_melds, CFG)
        for ext in exts:
            extensions_found.append((card, ext))
    if extensions_found:
        print('Extensions possible for RAMAI:')
        for card, ext in extensions_found:
            target = 'your' if ext.meld_owner == 0 else 'RAMAI\'s'
            print(f'  {card.name} → extends {target} meld {ext.meld_index} ({ext.extends_at})')
    else:
        print('No extensions possible.')

## Cell 11 — End of game analysis

In [ ]:
print('=' * 60)
print('GAME ANALYSIS')
print('=' * 60)
print()

if state.winner is not None:
    winner = 'You' if state.winner == 0 else 'RAMAI'
    print(f'Winner: {winner}')
else:
    print('Game not finished. Continue (Cells 7-8).')

print(f'Turns played: {state.turn}')
print()
print('Melds laid by you:')
for i, m in enumerate(state.players[0].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
print()
print('Melds laid by RAMAI:')
for i, m in enumerate(state.players[1].laid_melds):
    print(f'  {i+1}. {" ".join(c.name for c in m)}')
    desigs = designate_jokers(m, CFG)
    for d in desigs:
        print(f'     {d.name}')

print()
print('Final card counting:')
for p in range(CFG.num_players):
    name = 'You' if p == 0 else 'RAMAI'
    h = counting.hand_count(p)
    print(f'  {name} hand (inferred): {h} cards')

## Cell 12 — Run tests (verification)

In [ ]:
!cd {REPO} && python -m pytest tests/ 2>&1 | tail -5

---

## Appendix — MANUAL mode (debug fallback)

If the camera is unavailable (no getUserMedia, no file-input fallback), the notebook defaults to MANUAL mode. In this mode, ALL card state is entered via grids. This is a debug mode — the real project uses the camera.

## Appendix — Train your own YOLO cards model

If you want a CARDS-specific model (instead of the COCO backbone from `v0.1.0-vision-bootstrap`):

1. Open `notebooks/train_yolo.ipynb` in Colab (GPU runtime)
2. Run it — fine-tunes YOLOv8n on the Kaggle playing-cards dataset (~30 min)
3. Upload the resulting `best.pt` to a new GitHub release `v0.2.0-cards`
4. Update `DOWNLOAD_URL` in `rami/vision/pretrained.py` to point to the new release

The training notebook reports mAP50 on the test set. That number goes in the README.